In [2]:


# 2) Forzar Java 17 en este kernel
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# 3) Verifica que ahora ve Java 17
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)


In [4]:

# 5) Leer directo el gzip (no hace falta descomprimir)
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("fhvhv_tripdata_2021-01.csv.gz"))
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: integer (nullable = true)



In [6]:
#import pyspark
#from pyspark.sql import SparkSession
#spark = SparkSession.builder \
#    .master("local[*]") \
#    .appName('test') \
#    .getOrCreate()
#!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
#!gzip -dc fhvhv_tripdata_2021-01.csv.gz > fhvhv_tripdata_2021-01.csv
#!wc -l fhvhv_tripdata_2021-01.csv
#df = (spark.read
#    .option("header", "true")
#    .option("inferSchema", "true")
#    .csv("fhvhv_tripdata_2021-01.csv.gz"))
#df.printSchema()

In [7]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [10]:
df_pandas = pd.read_csv('head.csv')

In [12]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [13]:
from pyspark.sql import types

In [15]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [17]:
df = df.repartition(24)

In [20]:
df.write.parquet('fhvhv/2021/01/')

In [22]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [23]:
from pyspark.sql import functions as F

In [27]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [28]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(F.col('PULocationID') == 132) \
    .show(5)

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-06 19:23:50|2021-01-06 19:47:43|         132|          80|
|2021-01-06 15:19:18|2021-01-06 16:08:15|         132|          22|
|2021-01-02 13:15:42|2021-01-02 13:33:47|         132|         138|
|2021-01-01 11:02:45|2021-01-01 11:15:18|         132|         215|
|2021-01-01 15:06:15|2021-01-01 15:43:05|         132|         265|
+-------------------+-------------------+------------+------------+
only showing top 5 rows


In [30]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .take(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 5, 22, 14, 7), dropoff_datetime=datetime.datetime(2021, 1, 5, 22, 32, 28), PULocationID=189, DOLocationID=107),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 17, 59, 55), dropoff_datetime=datetime.datetime(2021, 1, 2, 18, 10, 39), PULocationID=88, DOLocationID=137),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 23, 57, 54), dropoff_datetime=datetime.datetime(2021, 1, 3, 0, 15, 48), PULocationID=238, DOLocationID=224),
 Row(pickup_datetime=datetime.datetime(2021, 1, 6, 15, 53, 13), dropoff_datetime=datetime.datetime(2021, 1, 6, 16, 7, 7), PULocationID=169, DOLocationID=208),
 Row(pickup_datetime=datetime.datetime(2021, 1, 7, 7, 35, 24), dropoff_datetime=datetime.datetime(2021, 1, 7, 7, 55, 49), PULocationID=75, DOLocationID=88)]

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(F.col('PULocationID') == 132) \
    .show(5)

In [34]:
df \
    .withColumn('pickup_date', F.to_date('pickup_datetime')) \
    .withColumn('dropoff_date', F.to_date('dropoff_datetime')) \
    .select('pickup_date', 'dropoff_date','PULocationID','DOLocationID') \
    .show(5)

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-03|  2021-01-03|         255|          34|
| 2021-01-05|  2021-01-05|         189|         107|
| 2021-01-02|  2021-01-02|          88|         137|
| 2021-01-02|  2021-01-03|         238|         224|
| 2021-01-06|  2021-01-06|         169|         208|
+-----------+------------+------------+------------+
only showing top 5 rows


In [36]:
crazy_stuff('B02884')

's/b44'

In [38]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-03|  2021-01-03|         255|          34|
|  e/b42| 2021-01-05|  2021-01-05|         189|         107|
|  e/b33| 2021-01-02|  2021-01-02|          88|         137|
|  e/b38| 2021-01-02|  2021-01-03|         238|         224|
|  e/b3b| 2021-01-06|  2021-01-06|         169|         208|
|  e/b33| 2021-01-07|  2021-01-07|          75|          88|
|  e/acc| 2021-01-07|  2021-01-07|         210|         210|
|  e/acc| 2021-01-02|  2021-01-02|         243|          69|
|  e/b35| 2021-01-04|  2021-01-04|         250|         213|
|  s/b3d| 2021-01-03|  2021-01-03|          87|          79|
|  e/a39| 2021-01-03|  2021-01-03|          68|         181|
|  s/acd| 2021-01-04|  2021-01-04|          95|         236|
|  s/b13| 2021-01-02|  2021-01-02|         262|         236|
|  e/9ce| 2021-01-04|  2

In [40]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
